In [2]:
import numpy as np

In [3]:
def op(row_or_col : str, type : str, data: list, L, M, R):
    '''Perform either row or column operations to the integer matrix M.
     Modifyies L, M, R in place so that the product LMR remains unchanged '''
    if row_or_col == 'row':
        # row operation
        if type == 'swap': # swap two rows
            i, j = data
            M[[i, j]] = M[[j, i]]
            L[[i, j]] = L[[j, i]]
        elif type == 'negate':
            i = data
            M[i] *= -1
            L[:, i] *= -1
        elif type == 'add':
            # row_i |-> row_i + k* row_j
            i, j, k = data
            M[i] = M[i] + k * M[j]
            L[:, j] = L[:, j] -k * L[:, i]
        elif type in {'bez', 'bezout'}:
            ''' Given x*n + y*m = gcd(m, m) =:d , multiply by the matrix
                    [  x    y  ]
                    [-m/d  n/d ]
            applied to the submatrix of rows i and j and identity elsewhere'''
            #-----------------------------------------------------------------
            # subroutine to find the gcd and bezout coefficients
            def bezout(n: int, m: int) -> list[int]:
                ''' Given two integers n and m, find integers x, y such that
                x*n + y*m = gcd(n, m)
                Returns [x, y, gcd(n, m)]'''    
                # Initialize: [x, y, value]
                x0, y0, r0 = 1, 0, n
                x1, y1, r1 = 0, 1, m                
                while r1 != 0: # do euclidean algorithm
                    q = r0 // r1
                    x0, x1 = x1, x0 - q * x1
                    y0, y1 = y1, y0 - q * y1
                    r0, r1 = r1, r0 - q * r1                
                return [x0, y0, r0]
            #------------------------------------------------------------------
            i, j, n, m = data
            x, y, d = bezout(n, m)
            a, b = n // d, m // d

            M[[i, j]] = x * M[i] + y* M[j], -b * M[i] + a * M[j]
            L[:, i], L[:, j] = (a * L[:, i] + b * L[:, j], 
                                -y * L[:, i] + x * L[:, j] )



        else:
            print('Error: that operation not supported')
    elif row_or_col in {'col', 'column'}:
        # take transpose and do row operations 
        op('row', type, data, R.T, M.T, L.T)
        


In [59]:
def eliminate(L, M, R, row=0, col=0):
    n, m = M.shape
    ''' given pivot in position (i, j), do row/col operations until 
    all other entries in row i and column j are zero'''
    def elim_row(L, M, R, row=0, col=0):
        for r in range(row + 1, n):
            pivot, other = M[row, col], M[r, col]
            if other % pivot == 0: # subtract off if multiple 
                q = other // pivot
                op('row', 'add', [r, row, -q], L, M, R)                    
            else: # use bezout operation to pass to gcd
                op('row', 'bez', [row, r, pivot, other], L, M, R)
                # now we can subtract off
                pivot, other = M[row, col], M[r, col]
                q = other // pivot
                op('row', 'add', [r, row, -q], L, M, R)

    def elim_col(L, M, R, row=0, col=0):
        for c in range(col + 1, m):
            pivot, other = M[row, col], M[row, c]
            if other % pivot == 0: # subtract off if multiple 
                q = other // pivot
                op('col', 'add', [c, col, -q], L, M, R)                    
            else: # use bezout operation to pass to gcd
                op('col', 'bez', [col, c, pivot, other], L, M, R)
                # now we can subtract off
                pivot, other = M[row, col], M[row, c]
                q = other // pivot
                op('col', 'add', [c, col, -q], L, M, R)
    def elim(L, M, R, row=0, col=0, type=0):
        if type % 2 == 0:
            elim_row(L, M, R, row, col)
        else: 
            elim_col(L, M, R, row, col)
    def done() -> bool:
        ith_row = M[row][row+1:]
        j_th_col = M[:,col][col+1:]
        return all(ith_row == 0) and all(j_th_col == 0)
    
    # keep repeating row/col/row/... elimination until all zeros
    iter = 0
    while not done():
        elim(L, M, R, row, col, iter)
        iter += 1

In [66]:
n, m = 3, 7
M = np.random.randint(-10, 10, (n, m)); M[0, 0] = 2
A = M.copy()
L = np.eye(n, dtype='i'); R = np.eye(m, dtype='i')
eliminate(L, A, R, 0, 0)
eliminate(L, A, R, 1, 1)
eliminate(L, A, R, 2, 2)
L, A, R, M, ((L@A@R - M).reshape(1,n*m)==0).all()


(array([[ -2,  -1,   0],
        [  3,   1,   0],
        [ 10, -85,   1]], dtype=int32),
 array([[-1,  0,  0,  0,  0,  0,  0],
        [ 0,  1,  0,  0,  0,  0,  0],
        [ 0,  0,  1,  0,  0,  0,  0]]),
 array([[    1,     4,    10,    12,     3,   -14,     0],
        [    0,     6,    21,    26,     3,   -35,    -1],
        [    0,   540,  1879,  2331,   284, -3121,   -91],
        [    0,  -221,  -769,  -954,     0,     0,     0],
        [    0,     0,     0,     0,     1,     0,     0],
        [    0,     0,     0,     0,     0,     1,     0],
        [    0,     0,     0,     0,     0,     0,     1]], dtype=int32),
 array([[  2,   2,  -1,  -2,   3,   7,   1],
        [ -3,  -6,  -9, -10,  -6,   7,  -1],
        [-10, -10,  -6,   1,  -1,  -6,  -6]]),
 True)

In [141]:
def smith_form(matrix):
    '''Given a matrix M with integer coefficients, returns a triple:
    [L, D, R] of matrices with integer coefficients, where L and R are 
    invertible over Z, D is diagonal and LDR = M. Uses recursive approach'''
    M = np.array(matrix, dtype='int64'); n, m = M.shape
    L = np.eye(n, dtype='int64')
    R = np.eye(m, dtype='int64')
    
    def setup_pivot(L, M, R, row=0) -> int:
        ''' Given array M, finds the first pivot column, swaps that row 
        to the top. Modifies the input arrays M and the array L keeps track
        of the row operations performed.
        '''
        # finds the pivot of the bottom right row x row submatrix
        n, m = M.shape
        for col in range(row, m):
            # Find pivot            
            pivot_row = None
            for other_row in range(row, n):
                if M[other_row, col] != 0:
                    pivot_row = other_row
                    break
            if pivot_row is None:
                continue

            # Swap pivot row into position
            if pivot_row != row:
                op('row', 'swap', [row, pivot_row], L, M, R)

            return col # returns the index of pivot column
        return None # no pivot found

    def eliminate(L, M, R, row=0, col=0):
        n, m = M.shape
        ''' given pivot in position (i, j), do row/col operations until 
        all other entries in row i and column j are zero'''
        def elim_row(L, M, R, row=0, col=0):
            pivot = M[row, col]
            for r in range(row + 1, n):
                pivot, other = M[row, col], M[r, col]
                if other % pivot == 0: # subtract off if multiple 
                    q = other // pivot
                    op('row', 'add', [r, row, -q], L, M, R)                    
                else: # use bezout operation to pass to gcd
                    op('row', 'bez', [row, r, pivot, other], L, M, R)
                    # now we can subtract off
                    pivot, other = M[row, col], M[r, col]
                    q = other // pivot
                    op('row', 'add', [r, row, -q], L, M, R)

        def elim_col(L, M, R, row=0, col=0):
            for c in range(col + 1, m):
                pivot, other = M[row, col], M[row, c]
                if other % pivot == 0: # subtract off if multiple 
                    q = other // pivot
                    op('col', 'add', [c, col, -q], L, M, R)                    
                else: # use bezout operation to pass to gcd
                    op('col', 'bez', [col, c, pivot, other], L, M, R)
                    # now we can subtract off
                    pivot, other = M[row, col], M[row, c]
                    q = other // pivot
                    op('col', 'add', [c, col, -q], L, M, R)
        def elim(L, M, R, row=0, col=0, type=0):
            if type % 2 == 0:
                elim_row(L, M, R, row, col)
            else: 
                elim_col(L, M, R, row, col)
        def done() -> bool:
            ith_row = M[row, col+1:]
            j_th_col = M[row+1:,col]
            return all(ith_row == 0) and all(j_th_col == 0)
        
        # keep repeating row/col/row/... elimination until all zeros
        iter = 0
        while not done():
            elim(L, M, R, row, col, iter)
            iter += 1

    for r in range(n):
        pivot_col = setup_pivot(L, M, R, r)
        if pivot_col != None:
            eliminate(L, M, R, r, pivot_col)
            # Make pivot positive
            if M[r, pivot_col] < 0:
                op('row', 'negate', r, L, M, R)
            if pivot_col == m:
                break
    return L, M, R


            



In [158]:
n, m = 4, 4
M = np.random.randint(-1, 1, (n, m))
A = M.copy()
L = np.eye(n, dtype='i'); R = np.eye(m, dtype='i')
L, M, R = smith_form(A)
L, M, R, (L@M@R - A), A


(array([[-1,  0,  0,  0],
        [ 0, -1,  0,  0],
        [ 0, -1, -1,  0],
        [ 0, -1,  0,  1]], dtype=int64),
 array([[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 0, 0]], dtype=int64),
 array([[1, 0, 1, 1],
        [0, 1, 1, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]], dtype=int64),
 array([[0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]], dtype=int64),
 array([[-1,  0, -1, -1],
        [ 0, -1, -1,  0],
        [ 0, -1, -1, -1],
        [ 0, -1, -1,  0]]))

In [138]:
i, j =2, 0
M = np.array([[1, 0,2 ], [0,1, 2],[7,8,9]])
M, M[i][i+1:] , M[:,j][j+1:], all(M[i][i+1:] == 0), all(M[:,j][j+1:]==0)

(array([[1, 0, 2],
        [0, 1, 2],
        [7, 8, 9]]),
 array([], dtype=int32),
 array([0, 7]),
 True,
 False)